___

# <font color= #99C8F5> **Tarea GARCH: TotalEnergies** </font>
#### <font color= #2E9AFE> `Modelos No Lineales Para Pronósticos`</font>
<Strong> Sofía Maldonado, Óscar Josué Rocha, Viviana Toledo & Isa Valladolid </Strong>

_15/03/2026._

___

### <font color= #99C8F5> **Sobre La Empresa** </font>

TotalEnergies (***TTE***) es una compañía petrolera fundada en 1924 en Francia. Es una de las grandes compañías petroleras privadas del mundo junto con ExxonMobil, Shell, Chevron, PB y Eni. Es la tercer compañía petrolera privada más grande del mundo tanto por ingresos como por ganancias, solamente detrás de ExxonMobil y Shell. La compañía tiene operaciones en 120 países. [1][2]

In [34]:
# Imports
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import acf, pacf
from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Descargar datos de TotalEnergies
ticker = 'TTE'
xom = yf.Ticker(ticker)
df = xom.history(start='2020-01-01', end='2026-03-13') # El 14 y el 15 es fin de semana

# Calculamos los retornos logarítmicos porcentuales usando 'Close'
returns = 100 * np.log(df['Close'] / df['Close'].shift(1)).dropna()

# Gráfica de la serie de retornos usando Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=returns.index, y=returns.squeeze(), mode='lines', name='Retornos TTE'))
fig.update_layout(title=f'Retornos Diarios de {ticker}',
                  yaxis_title='Retornos (%)')
fig.show()

### <font color= #99C8F5> **Selección de Modelo** </font>


In [36]:
# Retornos al cuadrado
sq_returns = returns**2

# Calcular ACF y PACF
lag_acf = acf(sq_returns, nlags=20)
lag_pacf = pacf(sq_returns, nlags=20, method='ols')

# Graficar ACF y PACF con Plotly
fig = make_subplots(rows=1, cols=2, subplot_titles=('ACF de Retornos al Cuadrado', 'PACF de Retornos al Cuadrado'))

# Añadir barras de ACF
fig.add_trace(go.Bar(x=np.arange(len(lag_acf)), y=lag_acf, name='ACF'), row=1, col=1)
# Añadir barras de PACF
fig.add_trace(go.Bar(x=np.arange(len(lag_pacf)), y=lag_pacf, name='PACF'), row=1, col=2)

# Líneas de significancia (aprox 1.96 / sqrt(N))
sig_level = 1.96 / np.sqrt(len(returns))
for i in [1, 2]:
    fig.add_hline(y=sig_level, line_dash="dash", line_color="red", row=1, col=i)
    fig.add_hline(y=-sig_level, line_dash="dash", line_color="red", row=1, col=i)

fig.update_layout(title='Análisis de Dependencia de Varianza', showlegend=False)
fig.show()

Sentimos que podría funcionar un modelo GARCH(1,1) para estos datos.

In [37]:
# Ajuste del modelo GARCH(1,1)
am = arch_model(returns, vol='Garch', p=1, q=1, dist='skewt')
res = am.fit(disp='off')


# Extraemos la volatilidad condicional modelada (Varianza que predijimos)
cond_vol = res.conditional_volatility

# Cálculo del Value at Risk (VaR) Histórico Condicional (GARCH) al 5%
# Cuantil empírico al 5% de los residuales estandarizados
q_5 = res.std_resid.quantile(0.05)

# VaR = Media Condicional + (Cuantil * Volatilidad Condicional)
# GARCH por defecto asume media constante, la extraemos de los parámetros
mean = res.params['mu']
VaR_5 = mean + cond_vol * q_5

# 5. Graficar los Retornos vs el VaR predictivo
fig = go.Figure()
fig.add_trace(go.Scatter(x=returns.index, y=returns.squeeze(), mode='lines',
                         name='Retornos Reales', opacity=0.6))
fig.add_trace(go.Scatter(x=VaR_5.index, y=VaR_5, mode='lines',
                         name='VaR 5% (GARCH)', line=dict(color='red')))

fig.update_layout(title='Backtesting de Riesgo: Retornos de TTE vs GARCH(1,1) VaR al 5%',
                  yaxis_title='Retornos (%)')
fig.show()

In [38]:
print(res.summary())

                           Constant Mean - GARCH Model Results                           
Dep. Variable:                             Close   R-squared:                       0.000
Mean Model:                        Constant Mean   Adj. R-squared:                  0.000
Vol Model:                                 GARCH   Log-Likelihood:               -2996.32
Distribution:      Standardized Skew Student's t   AIC:                           6004.64
Method:                       Maximum Likelihood   BIC:                           6036.73
                                                   No. Observations:                 1555
Date:                           Sun, Mar 15 2026   Df Residuals:                     1554
Time:                                   13:05:54   Df Model:                            1
                                 Mean Model                                
                 coef    std err          t      P>|t|     95.0% Conf. Int.
--------------------------------------

### <font color= #99C8F5> **Predicción de Volatilidad a Futuro** </font>

In [ ]:
preds = res.forecast(horizon=7, method='simulation', reindex=False)
variance = preds.variance

# extract last forecast row
pred = variance.iloc[-1]

# create index for next 7 periods
future_dates = pd.date_range(start=returns.index[-1], periods=8, freq='B')[1:]

pred_series = pd.Series(pred.values, index=future_dates)

preds.

Graficando...

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=returns.index, y=returns.squeeze(), mode='lines',
                         name='Retornos Reales', opacity=0.6))
fig.add_trace(go.Scatter(x=VaR_5.index, y=VaR_5, mode='lines',
                         name='VaR 5% (GARCH)', line=dict(color='red')))

fig.add_trace(go.Scatter(x=pred_series.index, y=pred_series, mode='lines', name='Predictions', line=dict(color='green')))

fig.update_layout(title='Predicciones',
                  yaxis_title='Retornos (%)')
fig.show()

### <font color= #99C8F5> **Half-Life Shock** </font>

El half-life shock, en el contexto de la volatilidad de una acción y los modelos GARCH, se refiere al tiempo requerido para que el efecto de un shock se disipe a la mitad (a la mitad ya que sería imposible calcular el tiempo para que se disipe completamente). [3] Este dato se puede conseguir con los parámetros $\alpha$ y $\beta$ de un modelo GARCH fitteado. La fórmula es la siguiente:

$$
\text{HL} = \frac{ln(0.5)}{ln(\alpha + \beta)}
$$

Esto se puede calcular fácilmente

In [59]:
alpha = res.params['alpha[1]']
beta = res.params['beta[1]']

hl = (np.log(0.5))/np.log(alpha+beta) # En NumPy, np.log() es el logaritmo natural

print(f'Half-Life Shock: {hl:.2f} dias.')

Half-Life Shock: 18.59 dias.


Se espera, entonces, que un shock se va a tomar poco más de 18 días en disiparse a la mitad.

### <font color= #99C8F5> **Referencias** </font>


[1] TotalEnergies. *Wikipedia*. https://totalenergies.com/company/identity/totalenergies-at-a-glance

[2] TotalEnergies, a Global Integrated Energy Company. *TotalEnergies*. https://totalenergies.com/company/identity/totalenergies-at-a-glance

[3] Doe, J. (2022). GARCH model: convergence of the conditional variance to the unconditional variance. *Cross Validated*. https://stats.stackexchange.com/questions/579696/garch-model-convergence-of-the-conditional-variance-to-the-unconditional-varian
